# Experiment 0.1.1 — local_234x2 direct Output WholeCount

Analysis-only notebook. This notebook reads finalized Exp0.1.1 artifacts and compares the new local `(234)->(234)->K` direct-SNN rows against Exp0.1. Training is performed only by the Slurm/Python experiment runner.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'notebooks' / 'artifacts' / 'experiment_0_1_1_local234_wholecount' / 'local234_wholecount_v1'
summary = pd.read_csv(RESULTS / 'summary.csv')
comparison = pd.read_csv(RESULTS / 'comparison_with_exp0_1.csv')
objective = pd.read_csv(RESULTS / 'paired_objective_effects.csv')
capacity = pd.read_csv(RESULTS / 'paired_capacity_effects.csv')
summary


## New local_234x2 direct-SNN conditions

All four conditions use **Output WholeCount** as the deployment readout. The only factors are training objective (`whole_count_ce` vs `timestep_ce`) and hidden event capacity (`binary` vs `multi_h`).


In [ ]:
display_cols = [
    'architecture', 'objective', 'variant',
    'test_ba_mean', 'test_ba_std', 'test_ba_count',
]
summary[display_cols].sort_values('test_ba_mean', ascending=False)


In [ ]:
plot_df = summary.copy()
plot_df['condition'] = plot_df['objective'] + ' / ' + plot_df['variant']
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(plot_df['condition'], plot_df['test_ba_mean'], yerr=plot_df['test_ba_std'], capsize=4)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('local_234x2 direct SNN — Output WholeCount')
ax.tick_params(axis='x', rotation=25)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## Paired objective effect

Positive `delta_test_ba_whole_minus_timestep` means WholeCount-CE improves the same seed/capacity model relative to timestep-CE while the deployment readout stays fixed at Output WholeCount.


In [ ]:
objective.groupby('variant')['delta_test_ba_whole_minus_timestep'].agg(['mean', 'std', 'count'])


## Paired hidden-capacity effect

Positive `delta_test_ba_multi_h_minus_binary` means hidden event cap 31 improves over pure binary hidden spikes for the same objective and seed. Output neurons remain binary in both cases.


In [ ]:
capacity.groupby('objective')['delta_test_ba_multi_h_minus_binary'].agg(['mean', 'std', 'count'])


## Combined Exp0.1 comparison

`comparison_with_exp0_1.csv` contains the finalized Exp0.1 systems plus the four new direct `local_234x2` conditions. This makes it possible to compare direct Output WholeCount against the frozen-SNN Fixed250 Linear decoder and the raw Fixed250/Relative10 Linear references.


In [ ]:
comparison.sort_values('mean_test_ba', ascending=False)[
    ['system', 'family', 'architecture', 'objective', 'variant', 'mean_test_ba', 'sd_test_ba', 'n']
]
